In [1]:
# ============================================================
# D7 — Branch C: Deterministic Normalisation
# 0. Imports and frozen experimental configuration
# ============================================================
!pip -q install pymupdf pymupdf4llm

import json
import hashlib
import platform
import re
import sys
import unicodedata

from collections import Counter
from datetime import datetime
from pathlib import Path

import fitz
import pandas as pd
import pymupdf4llm

from google.colab import files

DOCUMENT_ID = "D7"
DOCUMENT_NAME = (
    "UK National Audit Office — Delivering STEM "
    "(science, technology, engineering and mathematics) "
    "skills for the economy"
)

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"

SOURCE_FORMAT = ".pdf"
EXPECTED_SOURCE_SHA256 = (
    "00cd2555312b220d7b4289144261aaba"
    "888336fb21b32b9ff53e96427a5f7eba"
)

EXPECTED_PAGE_COUNT = 12

# ------------------------------------------------------------
# Fixed Stage 1 expectations.
# Used only AFTER extraction for diagnostics / Stage 4 validation.
# They are NOT disclosed to the model.
# ------------------------------------------------------------

EXPECTED_RECORD_COUNT = 59

EXPECTED_CATEGORY_COUNTS = {
    "Key fact": 10,
    "Policy context": 3,
    "Policy finding": 7,
    "Education pipeline statistic": 24,
    "Government initiative": 9,
    "Recommendation": 6
}

EXPECTED_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Topic",
    "Value",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location"
]

STRING_OR_NULL_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Topic",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location"
]

NUMERIC_OR_NULL_FIELDS = ["Value"]

MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Topic",
    "Source Location"
]

ALLOWED_CATEGORIES = set(EXPECTED_CATEGORY_COUNTS)

GROUNDING_MARKERS = {
    "key_facts": "Key facts",
    "summary": "Summary",
    "government_intervention": "Government intervention",
    "key_findings": "Key findings",
    "education_pipeline":
        "The performance of the education pipeline in delivering STEM skills",
    "latest_initiatives":
        "The latest initiatives designed to enhance the development of STEM skills",
    "value_for_money_conclusion":
        "Conclusion on value for money",
    "recommendations":
        "Recommendations"
}

REPRESENTATIVE_MARKERS = [
    "990",
    "442,000",
    "700,000",
    "2.6%",
    "30.9%",
    "42%",
    "9.4%",
    "21.2%",
    "6.9%",
    "19.9%",
    "17.6%",
    "67",
    "2,500",
    "15,000",
    "428",
    "810",
    "330",
    "Configure the labour market intelligence",
    "Fully embed a more structured approach to STEM across government"
]

OUTPUT_DIR = Path("outputs_D7_branch_C")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PARENT_CHECK_PATH = (
    OUTPUT_DIR / "D7_branch_C_parent_B_equivalence_check.json"
)
NORMALISATION_CHECK_PATH = (
    OUTPUT_DIR / "D7_branch_C_normalisation_check.json"
)
REPRESENTATION_PATH = (
    OUTPUT_DIR / "D7_branch_C_normalised_markdown.md"
)
REPRESENTATION_METADATA_PATH = (
    OUTPUT_DIR / "D7_branch_C_representation_metadata.json"
)
PROMPT_PATH = (
    OUTPUT_DIR / "D7_branch_C_prompt.txt"
)
EXPERIMENT_METADATA_PRE_PATH = (
    OUTPUT_DIR / "D7_branch_C_experiment_metadata_pre.json"
)
PRECHECK_PATH = (
    OUTPUT_DIR / "D7_branch_C_pre_extraction_check.json"
)
RAW_RESPONSE_PATH = (
    OUTPUT_DIR / "D7_branch_C_raw_response.txt"
)
PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR / "D7_branch_C_parsed_extraction.json"
)
STRUCTURE_CHECK_PATH = (
    OUTPUT_DIR / "D7_branch_C_structure_check.json"
)
EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR / "D7_branch_C_experiment_metadata.json"
)
EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR / "D7_branch_C_experiment_summary.json"
)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Parent branch:", PARENT_BRANCH)
print("Expected pages:", EXPECTED_PAGE_COUNT)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 73.0 MB/s eta 0:00:00
Document: D7
Branch: C
Parent branch: B
Expected pages: 12


In [2]:
# ============================================================
# 1. Upload original D7 PDF and required Branch B artefacts
# ============================================================
# Upload exactly:
#   1) original D7 PDF
#   2) D7_branch_B_structural_markdown.md
#   3) D7_branch_B_conversion_integrity.json

uploaded = files.upload()
names = list(uploaded.keys())

pdf_files = [Path(name) for name in names if name.lower().endswith(".pdf")]
md_files = [Path(name) for name in names if name.lower().endswith(".md")]
json_files = [Path(name) for name in names if name.lower().endswith(".json")]

if len(pdf_files) != 1 or len(md_files) != 1 or len(json_files) != 1:
    raise ValueError(
        "Upload exactly one PDF, one Branch B structural Markdown file, "
        "and one Branch B conversion-integrity JSON file."
    )

SOURCE_PATH = pdf_files[0]
BRANCH_B_REPRESENTATION_PATH = md_files[0]
BRANCH_B_CHECK_PATH = json_files[0]

print("Source:", SOURCE_PATH.name)
print("Branch B representation:", BRANCH_B_REPRESENTATION_PATH.name)
print("Branch B integrity:", BRANCH_B_CHECK_PATH.name)


Saving D7_branch_B_conversion_integrity.json to D7_branch_B_conversion_integrity.json
Saving D7_branch_B_structural_markdown.md to D7_branch_B_structural_markdown.md
Saving D7 - UK National Audit Office – STEM Report.pdf to D7 - UK National Audit Office – STEM Report.pdf
Source: D7 - UK National Audit Office – STEM Report.pdf
Branch B representation: D7_branch_B_structural_markdown.md
Branch B integrity: D7_branch_B_conversion_integrity.json


In [3]:
# ============================================================
# 2. Verify frozen source identity and Branch B parent integrity
# ============================================================
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

if SOURCE_PATH.suffix.lower() != SOURCE_FORMAT:
    raise ValueError("Unexpected D7 source format.")

SOURCE_SHA256 = sha256_file(SOURCE_PATH)
SOURCE_HASH_MATCH = SOURCE_SHA256 == EXPECTED_SOURCE_SHA256

if not SOURCE_HASH_MATCH:
    raise ValueError(
        "Uploaded D7 PDF does not match the frozen Stage 1 source identity."
    )

pdf_document = fitz.open(SOURCE_PATH)
PAGE_COUNT = len(pdf_document)

if PAGE_COUNT != EXPECTED_PAGE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_PAGE_COUNT} pages; observed {PAGE_COUNT}."
    )

if not all((page.get_text("text") or "").strip() for page in pdf_document):
    raise ValueError(
        "D7 is expected to contain a machine-readable text layer. "
        "OCR is not introduced in Branch C."
    )

with open(BRANCH_B_CHECK_PATH, "r", encoding="utf-8") as f:
    branch_b_check = json.load(f)

if branch_b_check.get("document_id") != DOCUMENT_ID:
    raise ValueError("Branch B integrity artefact belongs to another document.")

if branch_b_check.get("branch") != "B":
    raise ValueError("Uploaded integrity artefact is not from Branch B.")

if branch_b_check.get("source_sha256") != SOURCE_SHA256:
    raise ValueError(
        "Branch B parent representation was generated from another source identity."
    )

if not branch_b_check.get("conversion_integrity_passed", False):
    raise ValueError(
        "Branch B parent representation did not pass conversion integrity."
    )

SOURCE_B_MARKDOWN = (
    BRANCH_B_REPRESENTATION_PATH.read_text(encoding="utf-8")
)

if not SOURCE_B_MARKDOWN.strip():
    raise ValueError("Uploaded Branch B structural Markdown is empty.")

SOURCE_B_SHA256 = sha256_text(SOURCE_B_MARKDOWN)

print("Frozen source identity verified.")
print("Branch B conversion integrity verified.")
print("Branch B SHA-256:", SOURCE_B_SHA256)


Frozen source identity verified.
Branch B conversion integrity verified.
Branch B SHA-256: cc547d17e6b93240ae32c9b90173bc7a2c9cc7dbc4d49b7b2b1edb763d9e1603


In [4]:
# ============================================================
# 3. Reproduce the exact Branch B complete structural representation
# ============================================================
try:
    page_chunks = pymupdf4llm.to_markdown(
        str(SOURCE_PATH),
        page_chunks=True,
        write_images=False,
        show_progress=True
    )
except Exception as exc:
    raise RuntimeError(
        "D7 Branch B reproduction failed. "
        "No fallback representation is used. "
        f"Original error: {exc}"
    )

if not isinstance(page_chunks, list):
    raise TypeError("Expected page_chunks=True to return a list.")

if len(page_chunks) != EXPECTED_PAGE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_PAGE_COUNT} converted page chunks; "
        f"observed {len(page_chunks)}."
    )

page_markdown = {}

for page_number, chunk in enumerate(page_chunks, start=1):
    text = chunk.get("text", "") if isinstance(chunk, dict) else str(chunk)

    if not text.strip():
        raise ValueError(
            f"Converted Markdown for source page {page_number} is empty."
        )

    page_markdown[page_number] = text.rstrip()

markdown_parts = [
    "# D7 — UK National Audit Office STEM Report",
    "",
    "> Complete structural conversion of the original 12-page PDF.",
    "> No pages have been removed from the Branch B representation.",
    ""
]

for page_number in range(1, EXPECTED_PAGE_COUNT + 1):
    markdown_parts.extend([
        f"## Source Page {page_number}",
        "",
        page_markdown[page_number],
        ""
    ])

REPRODUCED_BRANCH_B_MARKDOWN = (
    "\n".join(markdown_parts).rstrip()
    + "\n"
)

REPRODUCED_B_SHA256 = sha256_text(
    REPRODUCED_BRANCH_B_MARKDOWN
)

print("Reproduced Branch B SHA-256:", REPRODUCED_B_SHA256)
print("Converted pages:", len(page_markdown))


Parsing 12 pages of 'D7 - UK National Audit Office – STEM Report.pdf'...


100%|██████████| 12/12 [00:10<00:00,  1.18it/s]


=== Document parser messages ===
Using Tesseract for OCR processing.


Generating markdown text...


100%|██████████| 12/12 [00:00<00:00, 1356.50it/s]

Reproduced Branch B SHA-256: cc547d17e6b93240ae32c9b90173bc7a2c9cc7dbc4d49b7b2b1edb763d9e1603
Converted pages: 12


In [5]:
# ============================================================
# 4. Verify exact Branch B parent equivalence
# ============================================================
PARENT_EQUIVALENCE_PASSED = (
    REPRODUCED_BRANCH_B_MARKDOWN
    == SOURCE_B_MARKDOWN
)

parent_check = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "parent_branch": PARENT_BRANCH,
    "source_sha256": SOURCE_SHA256,

    "branch_B_conversion_integrity_passed":
        bool(branch_b_check.get("conversion_integrity_passed", False)),

    "uploaded_branch_B_sha256":
        SOURCE_B_SHA256,

    "reproduced_branch_B_sha256":
        REPRODUCED_B_SHA256,

    "branch_B_representation_exactly_reproduced":
        PARENT_EQUIVALENCE_PASSED,

    "source_page_count":
        PAGE_COUNT,

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED
}

PARENT_CHECK_PATH.write_text(
    json.dumps(
        parent_check,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    parent_check,
    ensure_ascii=False,
    indent=2
))

if not PARENT_EQUIVALENCE_PASSED:
    raise ValueError(
        "The uploaded Branch B representation does not exactly match "
        "the representation reproduced from the frozen D7 PDF."
    )


{
  "document_id": "D7",
  "branch": "C",
  "parent_branch": "B",
  "source_sha256": "00cd2555312b220d7b4289144261aaba888336fb21b32b9ff53e96427a5f7eba",
  "branch_B_conversion_integrity_passed": true,
  "uploaded_branch_B_sha256": "cc547d17e6b93240ae32c9b90173bc7a2c9cc7dbc4d49b7b2b1edb763d9e1603",
  "reproduced_branch_B_sha256": "cc547d17e6b93240ae32c9b90173bc7a2c9cc7dbc4d49b7b2b1edb763d9e1603",
  "branch_B_representation_exactly_reproduced": true,
  "source_page_count": 12,
  "parent_equivalence_passed": true
}


In [6]:
# ============================================================
# 5. Define conservative deterministic Branch C normalisation
# ============================================================
#
# Following D1–D6, Branch C changes representation only.
#
# Allowed:
# - Unicode NFKC
# - Unicode-space standardisation
# - typographic apostrophe standardisation
# - dash/minus-glyph standardisation
# - soft-hyphen removal
# - line-ending standardisation
# - horizontal whitespace normalisation
# - excessive blank-line standardisation
#
# Not applied:
# - page filtering/removal
# - semantic rewriting/harmonisation
# - unit conversion
# - numeric calculation
# - manual correction
# - reference-guided reconstruction
# ============================================================

UNICODE_SPACE_CHARACTERS = [
    "\u00a0", "\u1680", "\u2000", "\u2001", "\u2002",
    "\u2003", "\u2004", "\u2005", "\u2006", "\u2007",
    "\u2008", "\u2009", "\u200a", "\u202f", "\u205f",
    "\u3000"
]

APOSTROPHE_REPLACEMENTS = {
    "’": "'",
    "‘": "'",
    "‛": "'",
    "´": "'",
    "`": "'"
}

DASH_REPLACEMENTS = {
    "‐": "-",
    "‑": "-",
    "‒": "-",
    "–": "-",
    "—": "-",
    "−": "-"
}

def normalise_text_representation(text):
    text = unicodedata.normalize("NFKC", str(text))

    for character in UNICODE_SPACE_CHARACTERS:
        text = text.replace(character, " ")

    for source, target in APOSTROPHE_REPLACEMENTS.items():
        text = text.replace(source, target)

    for source, target in DASH_REPLACEMENTS.items():
        text = text.replace(source, target)

    text = text.replace("\u00ad", "")
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    lines = []

    for line in text.splitlines():
        line = re.sub(r"[ \t\f\v]+", " ", line).rstrip()
        lines.append(line)

    text = "\n".join(lines)

    # Preserve logical line structure; standardise only excessive
    # blank-line runs.
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip() + "\n"


In [7]:
# ============================================================
# 6. Apply Branch C normalisation to the COMPLETE Branch B representation
# ============================================================
NORMALISED_MARKDOWN = normalise_text_representation(
    SOURCE_B_MARKDOWN
)

if not NORMALISED_MARKDOWN.strip():
    raise ValueError(
        "D7 Branch C normalisation produced an empty representation."
    )

print("Branch B characters:", len(SOURCE_B_MARKDOWN))
print("Branch C characters:", len(NORMALISED_MARKDOWN))


Branch B characters: 23112
Branch C characters: 23017


In [8]:
# ============================================================
# 7. Verify Branch C normalisation integrity
# ============================================================
#
# Transformation-aware integrity check:
# raw-string equality is not required because Unicode/spacing
# normalisation is the intended Branch C intervention.
# ============================================================

# ------------------------------------------------------------
# A. Page-boundary sequence
# ------------------------------------------------------------

PAGE_PATTERN = re.compile(
    r"^## Source Page (\d+)$",
    flags=re.MULTILINE
)

parent_pages = PAGE_PATTERN.findall(
    SOURCE_B_MARKDOWN
)

branch_c_pages = PAGE_PATTERN.findall(
    NORMALISED_MARKDOWN
)

expected_pages = [
    str(page_number)
    for page_number in range(1, EXPECTED_PAGE_COUNT + 1)
]

page_sequence_preserved = (
    parent_pages
    == branch_c_pages
    == expected_pages
)

# ------------------------------------------------------------
# B. Exact deterministic transformation reproducibility
# ------------------------------------------------------------

EXPECTED_NORMALISED_MARKDOWN = normalise_text_representation(
    SOURCE_B_MARKDOWN
)

deterministic_representation_verified = (
    NORMALISED_MARKDOWN
    == EXPECTED_NORMALISED_MARKDOWN
)

# ------------------------------------------------------------
# C. Expected major source components
# ------------------------------------------------------------

canonical_representation = NORMALISED_MARKDOWN.casefold()

component_checks = {}

for key, marker in GROUNDING_MARKERS.items():
    canonical_marker = (
        normalise_text_representation(marker)
        .strip()
        .casefold()
    )

    component_checks[key] = (
        canonical_marker
        in canonical_representation
    )

all_expected_components_preserved = all(
    component_checks.values()
)

# ------------------------------------------------------------
# D. Representative fixed-scope markers
# ------------------------------------------------------------

representative_marker_checks = {}

for marker in REPRESENTATIVE_MARKERS:
    canonical_marker = (
        normalise_text_representation(marker)
        .strip()
        .casefold()
    )

    representative_marker_checks[marker] = (
        canonical_marker
        in canonical_representation
    )

all_representative_content_preserved = all(
    representative_marker_checks.values()
)

# ------------------------------------------------------------
# E. Transformation-aware quantitative token preservation
# ------------------------------------------------------------

VALUE_PATTERNS = {
    "currency_amounts":
        r"£\s*\d[\d,]*(?:\.\d+)?(?:\s*(?:m|million))?",

    "percentages":
        r"(?<![\w])\d+(?:\.\d+)?\s*%",

    "comma_separated_numbers":
        r"(?<![\w])\d{1,3}(?:,\d{3})+(?:\.\d+)?(?![\w])",

    "academic_years":
        r"\b20\d{2}/\d{2}\b",

    "plain_million_expressions":
        r"\b\d+(?:\.\d+)?\s+million\b"
}

def canonicalise_value_token(token):
    token = normalise_text_representation(token).strip()
    token = re.sub(r"[ \t]+", " ", token)
    token = re.sub(r"£\s+", "£", token)
    return token.casefold()

numeric_preservation = {}

for label, pattern in VALUE_PATTERNS.items():
    before = [
        canonicalise_value_token(token)
        for token in re.findall(
            pattern,
            SOURCE_B_MARKDOWN,
            flags=re.IGNORECASE
        )
    ]

    after = [
        canonicalise_value_token(token)
        for token in re.findall(
            pattern,
            NORMALISED_MARKDOWN,
            flags=re.IGNORECASE
        )
    ]

    before_counter = Counter(before)
    after_counter = Counter(after)

    missing = list(
        (before_counter - after_counter).elements()
    )

    added = list(
        (after_counter - before_counter).elements()
    )

    numeric_preservation[label] = {
        "count_before": len(before),
        "count_after": len(after),
        "missing_token_count": len(missing),
        "added_token_count": len(added),
        "passed": (
            len(missing) == 0
            and len(added) == 0
        )
    }

numeric_values_preserved = all(
    result["passed"]
    for result in numeric_preservation.values()
)

# ------------------------------------------------------------
# F. Physical extraction-scope pages must remain represented,
#    but the full 12-page document must also remain.
# ------------------------------------------------------------

fixed_scope_pages = list(range(6, 13))

scope_page_presence = {
    str(page_number):
        f"## Source Page {page_number}"
        in NORMALISED_MARKDOWN
    for page_number in fixed_scope_pages
}

all_scope_pages_present = all(
    scope_page_presence.values()
)

# ------------------------------------------------------------
# G. Final integrity decision
# ------------------------------------------------------------

normalisation_integrity_passed = bool(
    PARENT_EQUIVALENCE_PASSED
    and page_sequence_preserved
    and deterministic_representation_verified
    and all_expected_components_preserved
    and all_representative_content_preserved
    and numeric_values_preserved
    and all_scope_pages_present
)

normalisation_check = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "parent_branch": PARENT_BRANCH,

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "parent_page_markers":
        parent_pages,

    "branch_C_page_markers":
        branch_c_pages,

    "page_sequence_preserved":
        page_sequence_preserved,

    "deterministic_representation_verified":
        deterministic_representation_verified,

    "component_checks":
        component_checks,

    "all_expected_components_preserved":
        all_expected_components_preserved,

    "representative_content_checks":
        representative_marker_checks,

    "all_representative_content_preserved":
        all_representative_content_preserved,

    "numeric_token_preservation":
        numeric_preservation,

    "numeric_values_preserved":
        numeric_values_preserved,

    "fixed_scope_pages":
        fixed_scope_pages,

    "scope_page_presence":
        scope_page_presence,

    "all_scope_pages_present":
        all_scope_pages_present,

    "complete_12_page_representation_retained":
        True,

    "source_scope_filtering_applied":
        False,

    "page_removal_applied":
        False,

    "page_cropping_applied":
        False,

    "ocr_applied":
        False,

    "unicode_nfkc_normalisation_applied":
        True,

    "unicode_space_standardisation_applied":
        True,

    "apostrophe_standardisation_applied":
        True,

    "dash_and_minus_standardisation_applied":
        True,

    "soft_hyphen_removal_applied":
        True,

    "line_endings_standardised":
        True,

    "horizontal_whitespace_normalisation_applied":
        True,

    "paragraph_line_merging_applied":
        False,

    "line_break_hyphenation_repair_applied":
        False,

    "semantic_harmonisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "manual_correction_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "normalisation_integrity_passed":
        normalisation_integrity_passed
}

NORMALISATION_CHECK_PATH.write_text(
    json.dumps(
        normalisation_check,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    normalisation_check,
    ensure_ascii=False,
    indent=2
))

if not normalisation_integrity_passed:
    raise ValueError(
        "D7 Branch C normalisation-integrity checks failed. "
        "Inspect parent equivalence, page/component preservation "
        "and transformation-aware quantitative diagnostics."
    )


{
  "document_id": "D7",
  "branch": "C",
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "parent_page_markers": [
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8",
    "9",
    "10",
    "11",
    "12"
  ],
  "branch_C_page_markers": [
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8",
    "9",
    "10",
    "11",
    "12"
  ],
  "page_sequence_preserved": true,
  "deterministic_representation_verified": true,
  "component_checks": {
    "key_facts": true,
    "summary": true,
    "government_intervention": true,
    "key_findings": true,
    "education_pipeline": true,
    "latest_initiatives": true,
    "value_for_money_conclusion": true,
    "recommendations": true
  },
  "all_expected_components_preserved": true,
  "representative_content_checks": {
    "990": true,
    "442,000": true,
    "700,000": true,
    "2.6%": true,
    "30.9%": true,
    "42%": true,
    "9.4%": true,
    "21.2%": true,
    "6.9%": true,
    "19.9

In [9]:
# ============================================================
# 8. Save Branch C representation
# ============================================================
REPRESENTATION_PATH.write_text(
    NORMALISED_MARKDOWN,
    encoding="utf-8"
)

REPRESENTATION_SHA256 = sha256_file(
    REPRESENTATION_PATH
)

print("Saved:", REPRESENTATION_PATH.name)
print("Representation SHA-256:", REPRESENTATION_SHA256)


Saved: D7_branch_C_normalised_markdown.md
Representation SHA-256: 18754acd4e97b28bd83729cef668f8d4e60c9366ca66efda400e961eca181677


In [10]:
# ============================================================
# 9. Create controlled Branch C extraction prompt
# ============================================================
#
# This mirrors the final Branch B task and instruction strength.
# Expected 59-record/category counts are deliberately NOT disclosed.
# ============================================================

BRANCH_C_PROMPT = """You are an information extraction assistant.

Extract the policy and quantitative records represented within the
defined scope of the attached deterministically normalised structural
Markdown representation of the report:

“Delivering STEM (science, technology, engineering and mathematics)
skills for the economy”.

Treat the attached deterministically normalised structural Markdown
document as the only source of information.

The complete 12-page source representation is attached.

Include records from the following defined source regions:

1. Every primary Key Facts item represented in the “Key facts” section.
   Treat explanatory or comparative wording embedded within a Key Facts
   item as context for that item rather than as an additional standalone
   record.

2. The principal policy-context statements represented in Summary
   paragraphs 1, 3 and 4 concerning:
   - the definition of STEM;
   - the main STEM skills-development routes;
   - departmental responsibilities for STEM skills.

3. The principal policy findings represented in Summary paragraphs
   7 to 12 and the conclusion on value for money in paragraph 21.

4. Every explicitly stated quantitative observation in Summary
   paragraphs 13 to 17 that belongs to the defined education-pipeline
   extraction scope.

5. The explicitly represented government-initiative observations in
   Summary paragraphs 18 to 20 concerning:

   - T levels and their career routes;
   - national colleges focusing on STEM skills;
   - the qualification level targeted by institutes of technology;
   - the maths and physics teacher supply package;
   - the target for recruiting additional maths and physics teachers;
   - the target for improving the skills of non-specialist teachers;
   - returning teachers recruited by the return to teaching pilot;
   - the recruitment target for that pilot;
   - returning teachers who completed the training provided.

Do not create an additional observation from comparative wording that
only describes the relationship between two already represented
quantities, such as a recruited number relative to its stated target.

6. Each recommendation represented in recommendations 22(a) to 24(f).

Exclude:

- publication metadata;
- contents-page entries;
- copyright and publisher information;
- contact details;
- website and social-media information;
- document prices;
- paragraph numbers and page numbers as observations;
- values appearing only in cross-references;
- qualitative explanatory details that are not one of the requested
  policy-context records, policy findings or recommendations;
- values that are not explicit source observations;
- calculated, derived or inferred values;
- Markdown representation metadata and page-boundary labels as
  observations.

For every included record extract exactly these fields:

- Category
- Statement or Section
- Metric
- Topic
- Value
- Unit
- Qualifier
- Reporting Period
- Source Location

Category:

Use exactly one of:

- Key fact
- Policy context
- Policy finding
- Education pipeline statistic
- Government initiative
- Recommendation

Statement or Section:

- Preserve a concise source-grounded statement or section label that
  identifies where the observation belongs.
- Do not introduce external interpretation.

Metric:

- Provide a concise source-grounded name for the quantitative metric
  or policy statement represented by the record.
- For qualitative policy records, use a concise label that identifies
  the policy concept or recommendation.

Topic:

- Preserve the relevant source-grounded STEM topic, population,
  programme, institution or policy area.
- Do not merge separate observations merely because they concern a
  similar topic.

Value:

- Use a JSON number for explicitly represented quantitative values.
- Use null for qualitative policy records that do not contain a
  primary quantitative value.
- Preserve explicitly negative values as negative numbers.
- When the source explicitly describes a quantitative change as a
  fall, decrease, decline or reduction, encode the Value as a negative
  number even when the printed percentage does not contain a minus sign.
- When the source explicitly describes a quantitative change as a rise,
  increase or growth, preserve the Value as positive.
- Do not calculate, derive, convert or infer values.
- Do not rescale, calculate or convert proportions or percentages.
  The directional-sign rule above is the only permitted sign encoding.

Unit:

- Preserve the source-grounded measurement unit.
- Use null when no explicit quantitative unit applies.
- Do not place approximation or inequality wording in Unit.

Qualifier:

- Preserve explicit approximation, inequality or threshold wording
  directly associated with a quantitative value, such as:
  “around”, “almost”, “over”, “more than”, “just over”, “minimum”
  or equivalent wording represented in the source.
- Use null when no explicit qualifier is associated with the value.

Reporting Period:

- Preserve explicitly associated years, academic years, durations,
  comparison periods or relative periods.
- Use null when no explicit reporting period applies.

Source Location:

Use concise physical-PDF source locations grounded in the source-page
boundaries represented in the Markdown, for example:

- PDF page 6 — Key facts
- PDF page 7 — Summary paragraph 1
- PDF page 7 — Summary paragraph 3
- PDF page 7 — Summary paragraph 4
- PDF page 8 — Summary paragraph 7
- PDF page 9 — Summary paragraph 8
- PDF page 9 — Summary paragraph 9
- PDF page 9 — Summary paragraph 10
- PDF page 9 — Summary paragraph 11
- PDF page 9 — Summary paragraph 12
- PDF page 10 — Summary paragraph 13
- PDF page 10 — Summary paragraph 14
- PDF page 10 — Summary paragraph 15
- PDF page 10 — Summary paragraph 16
- PDF page 11 — Summary paragraph 17
- PDF page 11 — Summary paragraph 18
- PDF page 11 — Summary paragraph 19
- PDF page 11 — Summary paragraph 20
- PDF page 11 — Summary paragraph 21
- PDF page 12 — Recommendation 22(a)
- PDF page 12 — Recommendation 22(b)
- PDF page 12 — Recommendation 23(c)
- PDF page 12 — Recommendation 23(d)
- PDF page 12 — Recommendation 24(e)
- PDF page 12 — Recommendation 24(f)

Additional extraction rules:

- Use the explicit structural cues and textual content represented in
  the Markdown.
- Preserve the physical PDF page references exposed by the source-page
  boundaries.
- Preserve repeated observations when the same or similar statistic
  is explicitly represented in different source sections.
- Do not deduplicate distinct source observations.
- Do not use external knowledge.
- Do not follow hyperlinks.
- Do not silently correct values, units or wording.
- Do not infer missing observations.
- Verify that all content within the defined source scope has been
  processed.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names and field order defined below.

Expected JSON structure:

{
  "document_id": "D7",
  "branch": "C",
  "records": [
    {
      "Category": null,
      "Statement or Section": null,
      "Metric": null,
      "Topic": null,
      "Value": null,
      "Unit": null,
      "Qualifier": null,
      "Reporting Period": null,
      "Source Location": null
    }
  ]
}

Return only the JSON object.
""".strip()

PROMPT_PATH.write_text(
    BRANCH_C_PROMPT,
    encoding="utf-8"
)

PROMPT_SHA256 = sha256_file(
    PROMPT_PATH
)

print(BRANCH_C_PROMPT)
print("Prompt SHA-256:", PROMPT_SHA256)


You are an information extraction assistant.

Extract the policy and quantitative records represented within the
defined scope of the attached deterministically normalised structural
Markdown representation of the report:

“Delivering STEM (science, technology, engineering and mathematics)
skills for the economy”.

Treat the attached deterministically normalised structural Markdown
document as the only source of information.

The complete 12-page source representation is attached.

Include records from the following defined source regions:

1. Every primary Key Facts item represented in the “Key facts” section.
   Treat explanatory or comparative wording embedded within a Key Facts
   item as context for that item rather than as an additional standalone
   record.

2. The principal policy-context statements represented in Summary
   paragraphs 1, 3 and 4 concerning:
   - the definition of STEM;
   - the main STEM skills-development routes;
   - departmental responsibilities for STEM sk

In [11]:
# ============================================================
# 10. Create representation and pre-extraction metadata
# ============================================================
REPRESENTATION_METADATA = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "parent_branch": PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "parent_B_representation_file":
        BRANCH_B_REPRESENTATION_PATH.name,

    "parent_B_representation_sha256":
        SOURCE_B_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "representation_type":
        "Complete Branch B structural Markdown with deterministic normalisation",

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "complete_12_page_representation_retained":
        True,

    "fixed_extraction_scope_pages":
        list(range(6, 13)),

    "scope_enforced_by_prompt_not_representation_filtering":
        True,

    "structural_conversion_inherited_from_branch_B":
        True,

    "normalisation_applied":
        True,

    "normalisation_operations": [
        "Unicode NFKC normalisation",
        "Unicode-space standardisation",
        "apostrophe standardisation",
        "dash/minus-glyph standardisation",
        "soft-hyphen removal",
        "line-ending standardisation",
        "horizontal whitespace normalisation",
        "excessive blank-line standardisation"
    ],

    "paragraph_line_merging_applied":
        False,

    "line_break_hyphenation_repair_applied":
        False,

    "semantic_harmonisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "manual_correction_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "normalisation_integrity_passed":
        normalisation_check["normalisation_integrity_passed"]
}

REPRESENTATION_METADATA_PATH.write_text(
    json.dumps(
        REPRESENTATION_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

EXPERIMENT_METADATA_PRE = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_HASH_MATCH,

    "input_representation":
        "Complete deterministically normalised structural Markdown",

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check["normalisation_integrity_passed"],

    "direct_document_ingestion":
        False,

    "structural_conversion_applied":
        True,

    "structural_conversion_inherited_from_branch_B":
        True,

    "normalisation_applied":
        True,

    "complete_source_document_retained":
        True,

    "source_scope_filtering_applied":
        False,

    "reference_values_disclosed_to_model":
        False,

    "reference_values_used_for_transformation":
        False,

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "manual_response_repair_permitted":
        False,

    "expected_output_format":
        "JSON object",

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "execution_environment":
        "Independent ChatGPT conversation",

    "model":
        "GPT-5.5",

    "created_at":
        datetime.now().isoformat(),

    "python_version":
        sys.version,

    "platform":
        platform.platform(),

    "validation_status":
        "Pending independent Branch C extraction and Stage 4 Validation C"
}

EXPERIMENT_METADATA_PRE_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    EXPERIMENT_METADATA_PRE,
    ensure_ascii=False,
    indent=2
))


{
  "document_id": "D7",
  "document_name": "UK National Audit Office — Delivering STEM (science, technology, engineering and mathematics) skills for the economy",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D7 - UK National Audit Office – STEM Report.pdf",
  "source_sha256": "00cd2555312b220d7b4289144261aaba888336fb21b32b9ff53e96427a5f7eba",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised structural Markdown",
  "representation_file": "D7_branch_C_normalised_markdown.md",
  "representation_sha256": "18754acd4e97b28bd83729cef668f8d4e60c9366ca66efda400e961eca181677",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "direct_document_ingestion": false,
  "structural_conversion_applied": true,
  "structural_conversion_inherited_from_branch_B": true,
  "normalisation_applied": true,
  "complete_source_document_retained": true,
  "source_scope_filtering

In [12]:
# ============================================================
# 11. Final pre-extraction control check
# ============================================================
PRECHECK = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "source_identity_verified":
        SOURCE_HASH_MATCH,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check["normalisation_integrity_passed"],

    "complete_12_page_representation_retained":
        True,

    "page_sequence_preserved":
        page_sequence_preserved,

    "all_scope_pages_present":
        all_scope_pages_present,

    "representation_exists":
        REPRESENTATION_PATH.exists(),

    "prompt_exists":
        PROMPT_PATH.exists(),

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "reference_values_used_for_transformation":
        False,

    "ready_for_independent_llm_execution": bool(
        SOURCE_HASH_MATCH
        and PARENT_EQUIVALENCE_PASSED
        and normalisation_check["normalisation_integrity_passed"]
        and REPRESENTATION_PATH.exists()
        and PROMPT_PATH.exists()
    )
}

PRECHECK_PATH.write_text(
    json.dumps(
        PRECHECK,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    PRECHECK,
    ensure_ascii=False,
    indent=2
))

if not PRECHECK["ready_for_independent_llm_execution"]:
    raise ValueError(
        "D7 Branch C is not ready for independent LLM execution."
    )


{
  "document_id": "D7",
  "branch": "C",
  "source_identity_verified": true,
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "complete_12_page_representation_retained": true,
  "page_sequence_preserved": true,
  "all_scope_pages_present": true,
  "representation_exists": true,
  "prompt_exists": true,
  "expected_record_count_disclosed_to_model": false,
  "expected_category_counts_disclosed_to_model": false,
  "reference_values_used_for_transformation": false,
  "ready_for_independent_llm_execution": true
}


In [13]:
# ============================================================
# 12. Download pre-extraction Branch C artefacts
# ============================================================
for path in [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    REPRESENTATION_METADATA_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    PRECHECK_PATH
]:
    files.download(path)

print(
    "\nIndependent extraction instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D7_branch_C_normalised_markdown.md.\n"
    "3. Submit D7_branch_C_prompt.txt exactly once.\n"
    "4. Do not upload the original PDF, Branch B artefacts, Stage 1 "
    "reference values, or previous extraction outputs.\n"
    "5. Do not manually repair, correct, or regenerate the response.\n"
    "6. Save the complete response exactly as returned in a plain-text file."
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Independent extraction instructions:
1. Open a new independent ChatGPT conversation.
2. Upload ONLY D7_branch_C_normalised_markdown.md.
3. Submit D7_branch_C_prompt.txt exactly once.
4. Do not upload the original PDF, Branch B artefacts, Stage 1 reference values, or previous extraction outputs.
5. Do not manually repair, correct, or regenerate the response.
6. Save the complete response exactly as returned in a plain-text file.


In [14]:
# ============================================================
# 13. Upload and preserve the complete raw Branch C response
# ============================================================
uploaded_response = files.upload()

if len(uploaded_response) != 1:
    raise ValueError(
        "Upload exactly one complete raw Branch C response file."
    )

RAW_RESPONSE_SOURCE = Path(
    next(iter(uploaded_response))
)

RAW_RESPONSE_TEXT = RAW_RESPONSE_SOURCE.read_text(
    encoding="utf-8"
)

RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)

RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)

print("Raw response preserved unchanged.")
print("Raw response SHA-256:", RAW_RESPONSE_SHA256)


Saving D7_branch_C_raw_response.txt to D7_branch_C_raw_response.txt
Raw response preserved unchanged.
Raw response SHA-256: eb33eecde3c2e34f5c47fa722459400ab3f379ff253420ff5b84628d09a35a8e


In [15]:
# ============================================================
# 14. Parse the raw response WITHOUT repair
# ============================================================
valid_json = True
json_parsing_error = None
parsed_response = None

try:
    parsed_response = json.loads(
        RAW_RESPONSE_TEXT
    )
except json.JSONDecodeError as exc:
    valid_json = False
    json_parsing_error = str(exc)

top_level_object_valid = (
    valid_json
    and isinstance(parsed_response, dict)
)

document_id_present = (
    top_level_object_valid
    and "document_id" in parsed_response
)

document_id_correct = (
    document_id_present
    and parsed_response.get("document_id") == DOCUMENT_ID
)

branch_present = (
    top_level_object_valid
    and "branch" in parsed_response
)

branch_correct = (
    branch_present
    and parsed_response.get("branch") == BRANCH
)

records_present = (
    top_level_object_valid
    and "records" in parsed_response
)

records_is_list = (
    records_present
    and isinstance(parsed_response.get("records"), list)
)

records_evaluable = bool(
    valid_json
    and top_level_object_valid
    and document_id_correct
    and branch_correct
    and records_is_list
)

extracted_records = (
    parsed_response["records"]
    if records_evaluable
    else []
)

observed_record_count = (
    len(extracted_records)
    if records_evaluable
    else None
)

print("Valid JSON:", valid_json)
print("Records evaluable:", records_evaluable)
print("Observed records:", observed_record_count)

if json_parsing_error:
    print("JSON parsing error:", json_parsing_error)


Valid JSON: True
Records evaluable: True
Observed records: 60


In [16]:
# ============================================================
# 15. Validate record schema and field types
# ============================================================
record_structure_issues = []
field_type_issues = []
missing_mandatory_values = []

if records_evaluable:

    for record_index, record in enumerate(extracted_records):

        if not isinstance(record, dict):
            record_structure_issues.append({
                "record_index": record_index,
                "issue": "Record is not a JSON object"
            })
            continue

        observed_fields = list(record.keys())

        if observed_fields != EXPECTED_FIELDS:
            record_structure_issues.append({
                "record_index": record_index,
                "issue": "Field names or field order differ",
                "expected_fields": EXPECTED_FIELDS,
                "observed_fields": observed_fields,
                "missing_fields": [
                    field
                    for field in EXPECTED_FIELDS
                    if field not in record
                ],
                "extra_fields": [
                    field
                    for field in observed_fields
                    if field not in EXPECTED_FIELDS
                ]
            })

        for field in STRING_OR_NULL_FIELDS:
            value = record.get(field)

            if (
                value is not None
                and not isinstance(value, str)
            ):
                field_type_issues.append({
                    "record_index": record_index,
                    "field": field,
                    "observed_type": type(value).__name__
                })

        value = record.get("Value")

        if (
            isinstance(value, bool)
            or (
                value is not None
                and not isinstance(value, (int, float))
            )
        ):
            field_type_issues.append({
                "record_index": record_index,
                "field": "Value",
                "observed_type": type(value).__name__
            })

        for field in MANDATORY_CONTENT_FIELDS:
            value = record.get(field)

            if value is None or (
                isinstance(value, str)
                and not value.strip()
            ):
                missing_mandatory_values.append({
                    "record_index": record_index,
                    "field": field
                })

record_schema_valid = (
    len(record_structure_issues) == 0
    if records_evaluable
    else None
)

field_types_valid = (
    len(field_type_issues) == 0
    if records_evaluable
    else None
)

mandatory_fields_complete = (
    len(missing_mandatory_values) == 0
    if records_evaluable
    else None
)

records_with_type_issues = len({
    issue["record_index"]
    for issue in field_type_issues
}) if records_evaluable else None

print("Record schema valid:", record_schema_valid)
print("Field types valid:", field_types_valid)
print("Mandatory fields complete:", mandatory_fields_complete)
print("Structure issues:", len(record_structure_issues))
print("Type issues:", len(field_type_issues))


Record schema valid: True
Field types valid: True
Mandatory fields complete: True
Structure issues: 0
Type issues: 0


In [17]:
# ============================================================
# 16. Content/scope diagnostics kept separate from schema validity
# ============================================================
if records_evaluable:

    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )

    observed_category_counts = dict(
        Counter(
            record.get("Category")
            for record in extracted_records
            if isinstance(record, dict)
        )
    )

    categories_valid = set(
        observed_category_counts
    ).issubset(ALLOWED_CATEGORIES)

    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )

    # Exact-complete-record duplicate diagnostic only.
    duplicate_counter = Counter(
        tuple(
            record.get(field)
            for field in EXPECTED_FIELDS
        )
        for record in extracted_records
        if isinstance(record, dict)
    )

    duplicate_complete_records = [
        {
            "record": list(key),
            "occurrence_count": count
        }
        for key, count in duplicate_counter.items()
        if count > 1
    ]

    duplicate_complete_record_count = len(
        duplicate_complete_records
    )

    negative_values = [
        record.get("Value")
        for record in extracted_records
        if (
            isinstance(record, dict)
            and isinstance(record.get("Value"), (int, float))
            and not isinstance(record.get("Value"), bool)
            and record.get("Value") < 0
        )
    ]

    expected_negative_values_present = all(
        value in negative_values
        for value in [-30.9, -30, -47]
    )

    null_value_count = sum(
        1
        for record in extracted_records
        if (
            isinstance(record, dict)
            and record.get("Value") is None
        )
    )

    qualifier_count = sum(
        1
        for record in extracted_records
        if (
            isinstance(record, dict)
            and record.get("Qualifier") is not None
        )
    )

else:

    record_count_valid = None
    observed_category_counts = None
    categories_valid = None
    category_counts_valid = None
    duplicate_complete_records = None
    duplicate_complete_record_count = None
    negative_values = None
    expected_negative_values_present = None
    null_value_count = None
    qualifier_count = None

scope_complete = bool(
    record_count_valid
    and category_counts_valid
) if records_evaluable else False

CONTENT_DIAGNOSTICS = {
    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches_reference":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "categories_valid":
        categories_valid,

    "category_counts_match_reference":
        category_counts_valid,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_value_count":
        len(missing_mandatory_values)
        if records_evaluable
        else None,

    "duplicate_complete_record_count":
        duplicate_complete_record_count,

    "duplicate_complete_records":
        duplicate_complete_records,

    "duplicate_check_is_diagnostic_only":
        True,

    "negative_values":
        negative_values,

    "expected_negative_values_present":
        expected_negative_values_present,

    "null_value_count":
        null_value_count,

    "qualifier_count":
        qualifier_count
}

print(json.dumps(
    CONTENT_DIAGNOSTICS,
    ensure_ascii=False,
    indent=2
))


{
  "expected_record_count": 59,
  "observed_record_count": 60,
  "record_count_matches_reference": false,
  "expected_category_counts": {
    "Key fact": 10,
    "Policy context": 3,
    "Policy finding": 7,
    "Education pipeline statistic": 24,
    "Government initiative": 9,
    "Recommendation": 6
  },
  "observed_category_counts": {
    "Key fact": 10,
    "Policy context": 3,
    "Policy finding": 7,
    "Education pipeline statistic": 25,
    "Government initiative": 9,
    "Recommendation": 6
  },
  "categories_valid": true,
  "category_counts_match_reference": false,
  "mandatory_fields_complete": true,
  "missing_mandatory_value_count": 0,
  "duplicate_complete_record_count": 0,
  "duplicate_complete_records": [],
  "duplicate_check_is_diagnostic_only": true,
  "negative_values": [
    -30.9,
    -30,
    -47
  ],
  "expected_negative_values_present": true,
  "null_value_count": 17,
  "qualifier_count": 14
}


In [18]:
# ============================================================
# 17. Determine technical/schema validity
# ============================================================
#
# Record/category counts, missing mandatory content and duplicate
# diagnostics are NOT conditions for schema validity.
# ============================================================

structure_valid = bool(
    valid_json
    and top_level_object_valid
    and document_id_correct
    and branch_correct
    and records_is_list
    and record_schema_valid is True
    and field_types_valid is True
)

STRUCTURE_CHECK = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        "Complete deterministically normalised structural Markdown",

    "valid_json":
        bool(valid_json),

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        bool(top_level_object_valid),

    "document_id_present":
        bool(document_id_present),

    "document_id_correct":
        bool(document_id_correct),

    "branch_present":
        bool(branch_present),

    "branch_correct":
        bool(branch_correct),

    "records_present":
        bool(records_present),

    "records_is_list":
        bool(records_is_list),

    "records_evaluable":
        bool(records_evaluable),

    "record_schema_valid":
        record_schema_valid,

    "record_structure_issues":
        record_structure_issues if records_evaluable else None,

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "field_type_issues":
        field_type_issues if records_evaluable else None,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "structure_valid":
        bool(structure_valid),

    "scope_complete":
        bool(scope_complete)
}

STRUCTURE_CHECK_PATH.write_text(
    json.dumps(
        STRUCTURE_CHECK,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    STRUCTURE_CHECK,
    ensure_ascii=False,
    indent=2
))


{
  "document_id": "D7",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "input_representation": "Complete deterministically normalised structural Markdown",
  "valid_json": true,
  "json_parsing_error": null,
  "top_level_object_valid": true,
  "document_id_present": true,
  "document_id_correct": true,
  "branch_present": true,
  "branch_correct": true,
  "records_present": true,
  "records_is_list": true,
  "records_evaluable": true,
  "record_schema_valid": true,
  "record_structure_issues": [],
  "field_types_valid": true,
  "records_with_type_issues": 0,
  "field_type_issues": [],
  "content_diagnostics": {
    "expected_record_count": 59,
    "observed_record_count": 60,
    "record_count_matches_reference": false,
    "expected_category_counts": {
      "Key fact": 10,
      "Policy context": 3,
      "Policy finding": 7,
      "Education pipeline statistic": 24,
      "Government initiative": 9,
      "Recommendation": 6
    },
    "observed_category_counts"

In [19]:
# ============================================================
# 18. Preserve parsed extraction only when records are evaluable
# ============================================================
parsed_extraction_created = False
parsed_extraction_sha256 = None

if records_evaluable:

    canonical_extraction = {
        "document_id": DOCUMENT_ID,
        "branch": BRANCH,
        "records": extracted_records
    }

    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            canonical_extraction,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    parsed_extraction_sha256 = sha256_file(
        PARSED_EXTRACTION_PATH
    )

    parsed_extraction_created = True

    print(
        "Parsed extraction saved:",
        PARSED_EXTRACTION_PATH.name
    )

else:

    print(
        "No parsed extraction created because the preserved raw "
        "response does not contain an evaluable records structure."
    )


Parsed extraction saved: D7_branch_C_parsed_extraction.json


In [20]:
# ============================================================
# 19. Create final experiment metadata and summary
# ============================================================
EXPERIMENT_METADATA = {
    **EXPERIMENT_METADATA_PRE,

    "raw_response_file":
        RAW_RESPONSE_PATH.name,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "parsed_extraction_file":
        PARSED_EXTRACTION_PATH.name
        if parsed_extraction_created
        else None,

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "observed_record_count":
        observed_record_count,

    "observed_category_counts":
        observed_category_counts,

    "structure_check_file":
        STRUCTURE_CHECK_PATH.name,

    "structure_valid":
        bool(structure_valid),

    "notes": (
        "Branch C applies deterministic normalisation to the exact "
        "complete Branch B 12-page structural Markdown representation. "
        "No pages are removed based on extraction scope. Stage 1 "
        "reference values and expected counts are not supplied to the "
        "model. Accuracy is evaluated separately in Validation C."
    )
}

EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_HASH_MATCH,

    "input_representation":
        "Complete deterministically normalised structural Markdown",

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check["normalisation_integrity_passed"],

    "structural_conversion_inherited_from_branch_B":
        True,

    "normalisation_applied":
        True,

    "complete_source_document_retained":
        True,

    "source_scope_filtering_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "raw_response_preserved":
        True,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "valid_json":
        bool(valid_json),

    "records_evaluable":
        bool(records_evaluable),

    "record_schema_valid":
        record_schema_valid,

    "field_types_valid":
        field_types_valid,

    "structure_valid":
        bool(structure_valid),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "scope_complete":
        bool(scope_complete),

    "duplicate_complete_record_count":
        duplicate_complete_record_count,

    "expected_negative_values_present":
        expected_negative_values_present,

    "qualifier_count":
        qualifier_count,

    "parsed_extraction_created":
        parsed_extraction_created,

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "accuracy_validation_completed":
        False,

    "validation_status":
        (
            "Pending Stage 4 Branch C validation against the fixed Stage 1 "
            "reference dataset using Branch A-frozen D7 comparison rules"
            if records_evaluable
            else
            "Not content-evaluable because the preserved Branch C response "
            "does not contain an evaluable records structure"
        )
}

EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    EXPERIMENT_SUMMARY,
    ensure_ascii=False,
    indent=2
))


{
  "document_id": "D7",
  "document_name": "UK National Audit Office — Delivering STEM (science, technology, engineering and mathematics) skills for the economy",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D7 - UK National Audit Office – STEM Report.pdf",
  "source_sha256": "00cd2555312b220d7b4289144261aaba888336fb21b32b9ff53e96427a5f7eba",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised structural Markdown",
  "representation_file": "D7_branch_C_normalised_markdown.md",
  "representation_sha256": "18754acd4e97b28bd83729cef668f8d4e60c9366ca66efda400e961eca181677",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "structural_conversion_inherited_from_branch_B": true,
  "normalisation_applied": true,
  "complete_source_document_retained": true,
  "source_scope_filtering_applied": false,
  "reference_values_used_for_transformation": false,
  "expec

In [21]:
# ============================================================
# 20. Final artefact inventory and download
# ============================================================
artefacts = [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    REPRESENTATION_METADATA_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    PRECHECK_PATH,
    RAW_RESPONSE_PATH,
    STRUCTURE_CHECK_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]

if parsed_extraction_created:
    artefacts.append(
        PARSED_EXTRACTION_PATH
    )

print("Final D7 Branch C artefacts:")

for path in artefacts:
    print("-", path.name, "| exists:", path.exists())

for path in artefacts:
    if path.exists():
        files.download(path)


Final D7 Branch C artefacts:
- D7_branch_C_parent_B_equivalence_check.json | exists: True
- D7_branch_C_normalisation_check.json | exists: True
- D7_branch_C_normalised_markdown.md | exists: True
- D7_branch_C_representation_metadata.json | exists: True
- D7_branch_C_prompt.txt | exists: True
- D7_branch_C_experiment_metadata_pre.json | exists: True
- D7_branch_C_pre_extraction_check.json | exists: True
- D7_branch_C_raw_response.txt | exists: True
- D7_branch_C_structure_check.json | exists: True
- D7_branch_C_experiment_metadata.json | exists: True
- D7_branch_C_experiment_summary.json | exists: True
- D7_branch_C_parsed_extraction.json | exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>